# Personalized Conversational Advertisement Ranking & Recommendation System — Walkthrough

Text query + `user_id` → ranked ads (retrieval → Learning-to-Rank), then proof the model beats a naive baseline.

Run `python train.py` once first (creates `model/ranker.txt`).

In [1]:
import warnings; warnings.filterwarnings('ignore')
from src.splits import load_all, split_interactions
ads, users, interactions = load_all()
print('ads', ads.shape, '| users', users.shape, '| interactions', interactions.shape)
print('overall CTR =', round(interactions.clicked.mean(), 3))
ads.head()

ads (150, 5) | users (80, 3) | interactions (4206, 3)
overall CTR = 0.714


,ad_id,brand,category,price,description
0,1,Asics,running_shoes,6754,"Asics breathable trainers, priced at Rs 6754."
1,2,Fjallraven,backpacks,1338,"Fjallraven casual college backpack, priced at ..."
2,3,Blue Tokai,coffee,1458,"Blue Tokai medium roast ground coffee, priced ..."
3,4,Vincent Chase,sunglasses,8807,"Vincent Chase retro round sunglasses, priced a..."
4,5,Apple,smartwatches,38029,"Apple AMOLED display smartwatch, priced at Rs ..."


## Step 0 — the fake world
Users click more on ads matching their past categories and budget. That baked-in signal is what the model learns.

In [2]:
users.head()

,user_id,past_categories_bought,avg_budget
0,u1,"books,sunglasses",3316
1,u2,smartwatches,33513
2,u3,"laptops,sunglasses,books",41619
3,u4,headphones,6196
4,u5,backpacks,5237


## Step 1 — sentence → structured query
Claude if `ANTHROPIC_API_KEY` is set, otherwise a regex fallback.

In [3]:
from src.query_parser import parse_query
for q in ["I'm looking for running shoes under Rs 5000",
          "show me noise cancelling headphones below 3k",
          "need a gaming laptop"]:
    print(f'{q!r:55} -> {parse_query(q)}')

"I'm looking for running shoes under Rs 5000"           -> {'category': 'running_shoes', 'budget': 5000}
'show me noise cancelling headphones below 3k'          -> {'category': 'headphones', 'budget': 3000}
'need a gaming laptop'                                  -> {'category': 'laptops', 'budget': None}


## Step 2 — retrieve candidate ads (embeddings + FAISS)

In [4]:
from src.retrieval import AdRetriever
r = AdRetriever(ads)
print('embedding backend:', r.backend, '| FAISS index:', r.uses_faiss)
r.query('running shoes under Rs 5000', k=5, restrict_category='running_shoes')[
    ['ad_id','brand','category','price','semantic_sim']]

embedding backend: tfidf | FAISS index: True


,ad_id,brand,category,price,semantic_sim
0,146,Reebok,running_shoes,3193,0.449105
1,96,Reebok,running_shoes,8660,0.449105
2,30,Nike,running_shoes,4095,0.445443
3,18,Puma,running_shoes,7671,0.443362
4,9,Puma,running_shoes,8271,0.443362


## Steps 3–4 — feature table + LightGBM ranking
Same ads, two different users → different order. That's the personalization.

In [5]:
from pipeline import AdPipeline
pipe = AdPipeline()
print('--- u1 (no running-shoes history) ---')
pipe.run('I want running shoes under Rs 5000', 'u1');
print('\n--- u11 (has bought running shoes) ---')
pipe.run('I want running shoes under Rs 5000', 'u11');

--- u1 (no running-shoes history) ---

Step 1  query      : 'I want running shoes under Rs 5000'
        parsed     : {'category': 'running_shoes', 'budget': 5000}
        embed text : 'running shoes under Rs 5000'
Step 2  retrieved  : 7 candidate ads
Step 3-4 ranked for user u1:

 ad_id  brand      category  price  semantic_sim  category_match  price_fit  user_cat_clicks    score
    93  Asics running_shoes   4681      0.405690               0          0                0 0.008551
   117 Reebok running_shoes   3578      0.402149               0          0                0 0.008551
   100   Nike running_shoes   4323      0.011779               0          0                0 0.008203
   108  Asics running_shoes   3048      0.405690               0          1                0 0.007492
    97   Nike running_shoes   2939      0.144901               0          1                0 0.006648
    30   Nike running_shoes   4095      0.445443               0          0                0 0.005592
   1

## Steps 5–6 — offline metrics + simulated A/B test

In [6]:
import evaluate
metrics, ab = evaluate.main()

STEP 5 - Offline ranking metrics (held-out)


  method  NDCG@10  Precision@10  users_scored
baseline 0.828150      0.746617            78
   model 0.924597      0.804309            78



STEP 6 - Simulated A/B test (two-proportion z-test)
Group A (baseline): 39 users | CTR = 0.779
Group B (model)   : 39 users | CTR = 0.928

Group B showed a +19.1% relative CTR lift over Group A
  absolute lift = +14.9 pts | z = 4.16 | p = 0.000
  95% CI on absolute lift = [+8.0%, +21.7%]
  -> STATISTICALLY SIGNIFICANT at alpha=0.05


**Takeaway:** the LightGBM ranker beats the similarity-only baseline on NDCG@10 and Precision@10, and the simulated A/B test shows a statistically significant CTR lift (p < 0.05). The model learned the personalization signal the baseline is blind to.

## Validation on a recognized benchmark — MovieLens-100K

The project's own ranker (`src.ranker.train_lambdarank`) is validated on the standard
academic recsys benchmark (100k ratings, 943 users), using the leave-one-out + 99-negatives
protocol from *Neural Collaborative Filtering* (He et al., WWW 2017):

| method | HR@10 | NDCG@10 |
|---|---|---|
| popularity baseline | 0.409 | 0.229 |
| **LightGBM LambdaRank** | **0.449** | **0.248** |

→ **+8.0% NDCG@10** over the popularity baseline on real, public data.

This validates the *ranking* claim on data nobody invented. It does not exercise Steps 1-2
(query parsing, retrieval) — no public dataset pairs natural-language shopping queries with
priced ads, which is exactly why `data/` is synthetic.

Run it with `python -m benchmark.benchmark_movielens` (see `benchmark/RESULTS.md`).